In [0]:
%run "/Workspace/Users/iancarodrigues108@gmail.com/ETL_Arquitetura_Medalhao_BOI_GORDO/0.Config/config"

catalogo:  workspace
schemas:  bronze_economia silver_economia gold_economia


In [0]:
#importando o functions as F, window 

from pyspark.sql import functions as F, Window

df= spark.table(f"{CATALOG}.{SILVER}.economia")



In [0]:
# fazendo window function a função de janela utilizando a data para ordenar

w = Window.orderBy("data")

gold = (df
        .withColumn("ipca_ant", F.lag("ipca").over(w))
        .withColumn("boi_ant", F.lag("boi_gordo").over(w))
        .withColumn("variacao_ipca", (F.col("ipca") - F.col("ipca_ant"))/F.col("ipca_ant")*100)
        .withColumn("variacao_boi", (F.col("boi_gordo") - F.col("boi_ant"))/F.col("boi_ant")*100)
        .drop("ipca_ant", "boi_ant")
       )
display(gold)
# gravando o dataframe no delta lake

gold.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{GOLD}.insights")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data,ipca,boi_gordo,data_coleta,variacao_ipca,variacao_boi
2025-01-01,0.16,324.95,2026-07-22T17:48:56.750Z,null,null
2025-02-01,1.31,319.21,2026-07-22T17:48:56.750Z,718.7500000000001,-1.7664256039390707
2025-03-01,0.56,312.47,2026-07-22T17:48:56.750Z,-57.25190839694656,-2.1114626734751267
2025-04-01,0.43,323.96,2026-07-22T17:48:56.750Z,-23.214285714285722,3.677153006688626
2025-05-01,0.26,308.15,2026-07-22T17:48:56.750Z,-39.53488372093023,-4.88023212742314
2025-06-01,0.24,313.51,2026-07-22T17:48:56.750Z,-7.692307692307699,1.7394126237222178
2025-07-01,0.26,299.97,2026-07-22T17:48:56.750Z,8.333333333333341,-4.318841504258226
2025-08-01,-0.11,307.25,2026-07-22T17:48:56.750Z,-142.3076923076923,2.4269093576024177
2025-09-01,0.48,307.87,2026-07-22T17:48:56.750Z,-536.3636363636364,0.20179007323026998
2025-10-01,0.09,310.51,2026-07-22T17:48:56.750Z,-81.25000000000001,0.8575047909832028


In [0]:
%sql
SELECT * FROM workspace.gold_economia.insights LIMIT 20

data,ipca,boi_gordo,data_coleta,variacao_ipca,variacao_boi
2025-01-01,0.16,324.95,2026-07-22T17:48:56.750Z,null,null
2025-02-01,1.31,319.21,2026-07-22T17:48:56.750Z,718.7500000000001,-1.7664256039390707
2025-03-01,0.56,312.47,2026-07-22T17:48:56.750Z,-57.25190839694656,-2.1114626734751267
2025-04-01,0.43,323.96,2026-07-22T17:48:56.750Z,-23.214285714285722,3.677153006688626
2025-05-01,0.26,308.15,2026-07-22T17:48:56.750Z,-39.53488372093023,-4.88023212742314
2025-06-01,0.24,313.51,2026-07-22T17:48:56.750Z,-7.692307692307699,1.7394126237222178
2025-07-01,0.26,299.97,2026-07-22T17:48:56.750Z,8.333333333333341,-4.318841504258226
2025-08-01,-0.11,307.25,2026-07-22T17:48:56.750Z,-142.3076923076923,2.4269093576024177
2025-09-01,0.48,307.87,2026-07-22T17:48:56.750Z,-536.3636363636364,0.20179007323026998
2025-10-01,0.09,310.51,2026-07-22T17:48:56.750Z,-81.25000000000001,0.8575047909832028


In [0]:
%sql
CREATE OR REPLACE VIEW workspace.gold_economia.vw_gold_dashboard AS
SELECT * FROM workspace.gold_economia.insights;

In [0]:
%sql
select data, variacao_ipca, variacao_boi 
from workspace.gold_economia.vw_gold_dashboard; 

data,variacao_ipca,variacao_boi
2025-01-01,null,null
2025-02-01,718.7500000000001,-1.7664256039390707
2025-03-01,-57.25190839694656,-2.1114626734751267
2025-04-01,-23.214285714285722,3.677153006688626
2025-05-01,-39.53488372093023,-4.88023212742314
2025-06-01,-7.692307692307699,1.7394126237222178
2025-07-01,8.333333333333341,-4.318841504258226
2025-08-01,-142.3076923076923,2.4269093576024177
2025-09-01,-536.3636363636364,0.20179007323026998
2025-10-01,-81.25000000000001,0.8575047909832028


In [0]:
%sql
select variacao_ipca, variacao_boi 
from workspace.gold_economia.vw_gold_dashboard; 

variacao_ipca,variacao_boi
null,null
718.7500000000001,-1.7664256039390707
-57.25190839694656,-2.1114626734751267
-23.214285714285722,3.677153006688626
-39.53488372093023,-4.88023212742314
-7.692307692307699,1.7394126237222178
8.333333333333341,-4.318841504258226
-142.3076923076923,2.4269093576024177
-536.3636363636364,0.20179007323026998
-81.25000000000001,0.8575047909832028


In [0]:
%sql
-- correlação global 

SELECT corr(variacao_ipca, variacao_boi) AS correlacao_global 
FROM workspace.gold_economia.vw_gold_dashboard;

correlacao_global
-0.14888420640482683


In [0]:
%sql
--´media de ipca, boi e correlacao
SELECT 
  AVG(ipca) AS media_ipca_global,
  AVG(boi_gordo) AS media_boi_gordo_global,
  corr(variacao_ipca, variacao_boi) AS media_correlacao_global
FROM workspace.gold_economia.vw_gold_dashboard;

media_ipca_global,media_boi_gordo_global,media_correlacao_global
0.3491666666666667,314.1841666666666,-0.14888420640482683


In [0]:
%sql
-- criando destaque, classe de impacto economico

WITH base AS (
  SELECT 
    data,
    variacao_ipca,
    variacao_boi,
    ABS(variacao_ipca - variacao_boi) AS diferenca_abs
  FROM workspace.gold_economia.vw_gold_dashboard
),
limiar AS (
  SELECT PERCENTILE_APPROX(diferenca_abs, 0.5) AS mediana
  FROM base
)
SELECT 
  b.data,
  CASE 
    WHEN b.diferenca_abs <= l.mediana THEN 'Empate'
    WHEN b.variacao_ipca > b.variacao_boi THEN 'Inflação cresce mais'
    ELSE 'Preço do boi cresce mais'
  END AS destaque,
  CASE 
    WHEN b.diferenca_abs <= l.mediana THEN 'Baixa divergência'
    ELSE 'Alta divergência'
  END AS classe_impacto
FROM base b
CROSS JOIN limiar l;

data,destaque,classe_impacto
2025-01-01,Preço do boi cresce mais,Alta divergência
2025-02-01,Inflação cresce mais,Alta divergência
2025-03-01,Empate,Baixa divergência
2025-04-01,Empate,Baixa divergência
2025-05-01,Empate,Baixa divergência
2025-06-01,Empate,Baixa divergência
2025-07-01,Empate,Baixa divergência
2025-08-01,Preço do boi cresce mais,Alta divergência
2025-09-01,Preço do boi cresce mais,Alta divergência
2025-10-01,Empate,Baixa divergência
